Citations: Andrew Lucas: Ising formulations of many NP Problems. arXiv:1302.5843. 
https://arxiv.org/abs/1302.5843

Ising machine: Vertex Cover

NP-Hard: Given an undirected graph G = (V,E), what is the smallest number of vertices that
can be "covered" such that every edge is incident to a covered vertex. The
NP-Complete form is decision-based (does a vertex cover of size K exist?)

In [53]:
import numpy as np
import scienceplots
import matplotlib.pyplot as plt
import numba
import math

In [54]:
# spin vector of size N depicting undirected graph G = (V, E). 1 for vertex included in cover, 0 for vertex not included in cover
N = 100
def generate_spins(N):
    return np.random.choice(np.array([1, 0], dtype=np.int64), size=N) # 1D array, size. generates random sample

xalpha = generate_spins(N)

In [55]:
# adjacency matrix of size NxN depicting undirected graph G = (V, E)
# 0 represents no connection, 1 represents a connection. 
# diagonal is all zereos. 

p = 0.5
@numba.njit
def build_adjacency_matrix(N, p):
    adjacency_matrix = np.zeros((N, N), dtype=np.int64)
    for i in range(N):
        for j in range(i + 1, N):
            if np.random.rand() < p and i != j:
                # Only add an edge if i and j are not the same node
                # and the random condition is met
                adjacency_matrix[i, j] = 1
                adjacency_matrix[j, i] = 1
    return adjacency_matrix

adjacency_matrix = build_adjacency_matrix(N, p)

np.set_printoptions(threshold=np.inf)

print("Adjacency Matrix:")
print(adjacency_matrix)

Adjacency Matrix:
[[0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 1 1 0 1 1 1 0 0 0 1 0 0 1 0 0 0 0 0 1 1
  0 0 0 1 1 0 1 0 0 1 0 1 1 1 1 0 1 1 1 0 1 0 1 0 1 0 0 1 0 1 0 1 0 0 1 0
  1 0 1 0 0 1 0 1 1 1 1 0 0 1 1 0 0 1 1 0 0 1 0 1 1 0 1 1]
 [0 0 1 0 1 0 0 0 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 1 0 1 0 1 0 1 1
  1 1 0 1 1 1 1 0 0 1 1 1 0 0 1 1 0 0 1 1 0 0 0 1 0 1 0 0 0 1 1 1 1 0 0 0
  0 0 1 1 1 0 1 0 0 1 1 0 1 0 1 0 1 0 1 0 1 1 0 0 1 1 0 0]
 [0 1 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 1 1 1 0 0 1 1 0 1
  1 0 1 1 1 1 1 1 1 1 1 0 1 0 0 0 0 1 1 0 1 0 0 1 0 1 1 0 1 1 1 0 1 0 0 1
  1 0 1 1 0 1 1 1 1 0 1 0 1 1 0 1 1 1 0 0 1 0 1 1 0 0 1 0]
 [0 0 0 0 1 1 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1 1 1 0 1 0 0 1 0 0 1
  0 1 1 1 0 0 0 1 0 1 0 0 1 0 1 0 0 0 0 0 1 0 1 1 0 1 0 1 0 1 0 0 1 0 0 0
  1 0 0 0 1 1 0 1 1 1 0 1 0 1 1 1 1 1 1 0 1 1 1 1 0 1 0 0]
 [0 1 0 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 1 1 1 1 0 0 1 0 0 0 0 0 0 1 0 1 1 0
  1 1 0 1 1 0 1 0 0 0 0 1 0 1 0 0 0 1 1 0 1 0 0 0 0 1 1 1 0 0 1 1 1 1 1 0
  0 0 

In [56]:

def get_ha(adjacency_matrix, xalpha, A):
    h_a = 0
    # for each edge (i, j) where adj[i,j] == 1 and i < j:
    #     h_a += (1 - xalpha[i]) * (1 - xalpha[j])
    # h_a *= A
    for i in range(N):
        for j in range(i + 1, N):
            if adjacency_matrix[i, j] == 1:
                h_a += (1 - xalpha[i]) * (1 - xalpha[j])
    h_a *= A
    return h_a

In [57]:
def get_hb(xalpha, B):
    return B * sum(xalpha)

In [58]:
# H = H(a) + H(b) <- Ising Hamiltonian 
def calc_energy(adjacency_matrix, vector, A, B):
    ha = get_ha(adjacency_matrix, vector, A)
    hb = get_hb(vector, B)
    return ha + hb

In [59]:
def metropolis(adjacency_matrix, xalpha, A, B, T_start, steps):
    energy = calc_energy(adjacency_matrix, xalpha, A, B)
    initial_energy = energy.copy()  # Store the initial energy for reference
    for step in range(steps):  # Number of Metropolis steps
        # Flip the spin of a random vertex
        j = np.random.randint(0, N)
        xalpha[j] = 1 - xalpha[j]  # Flip the spin
        new_energy = calc_energy(adjacency_matrix, xalpha, A, B)
        delta_energy = new_energy - energy
        
        # Accept the new configuration with Metropolis criterion
        T = T_start * (1 - step / steps)
        if delta_energy < 0 or np.random.rand() < math.exp(-delta_energy / T):
            energy = new_energy  # Accept the new configuration
        else:
            xalpha[j] = 1 - xalpha[j]  # Revert the flip if not accepted
    return xalpha, energy, initial_energy

In [60]:
# Main function set up 
steps = 200
A = 3.0
B = 1.0
T = 1.0
initial_xalpha = xalpha.copy()  # Store the initial vertex cover for reference
final_xalpha, final_energy, initial_energy = metropolis(adjacency_matrix, xalpha, A, B, T, steps)
print ("Initial Energy:", initial_energy)
print ("Final Energy:", final_energy)   
print ("Initial Vertex Cover:", xalpha)
print ("Final Vertex Cover:", final_xalpha)


Initial Energy: 1621.0
Final Energy: 98.0
Initial Vertex Cover: [1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1
 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
Final Vertex Cover: [1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1
 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
